<a href="https://colab.research.google.com/github/aodm26/FlightDataUS/blob/main/Group4_Streaming_Flight_Delay_Dashboard_standalone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# US Flight Delay — Live Streaming Dashboard (standalone)

Interactive PySpark + ipywidgets + folium dashboard. Pick a **Mode** and a **Question**, then click the green button.

**Three modes:**
1. **CrowdStrike outage replay (Jul 17–25 2024, chronological)** — the July 2024 IT outage across Delta and three peers, streamed in 50 small batches of 500. Delta highlighted; the spike is *carrier* delay, not weather.
2. **Overall sample stream (time-of-day)** — a random 10,000-flight sample, 10 batches, 17 panels across Volume · Delay · Carrier · Spatial · Anomaly.
3. **Analyse full dataset (one shot)** — exact aggregation over all ~7.08M flights for the selected question.

Delay **rates** use *operated* flights (cancelled/diverted excluded); cancellations are a separate KPI.

### Run
1. **Java 17** (auto-detected on macOS/Windows/Linux; if missing: `brew install openjdk@17`, Temurin 17 from adoptium.net, or `sudo apt install openjdk-17-jdk`).
2. `pip install pyspark==3.5.1 folium ipywidgets matplotlib pandas numpy`
3. **Edit `FILE_PATH`** in the *Load* cell to your `dataset.parquet`.
4. Run all cells top to bottom.

**To stop a running stream:** use the notebook toolbar's **Interrupt the kernel** button (the square ■ icon). The stream renders on the kernel thread so it displays reliably; that means the in-panel controls can't interrupt mid-run, but Interrupt always does.

In [1]:
# ==========================================================
# IMPORTS
# ==========================================================
import os, sys, time
import numpy as np
import pandas as pd

import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

import folium
from folium.plugins import HeatMap

# ==========================================================
# SPARK SETUP  (local mode — runs on macOS, Windows or Linux)
# ==========================================================
# ---- Point Spark at a Java 17 runtime (PySpark 3.5 needs Java 17) ----
# Auto-detects the OS, so the same notebook works on Mac and Windows.
import glob, platform, subprocess

def _java17_home():
    # 1) already set and valid? trust it
    jh = os.environ.get("JAVA_HOME")
    if jh and os.path.isdir(jh):
        return jh
    system = platform.system()
    if system == "Darwin":                                   # macOS
        try:
            return subprocess.check_output(
                ["/usr/libexec/java_home", "-v", "17"], text=True).strip()
        except Exception:
            pass
        cands = ["/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home",
                 "/usr/local/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"]
    elif system == "Windows":
        cands = (glob.glob(r"C:\Program Files\Java\jdk-17*") +
                 glob.glob(r"C:\Program Files\Eclipse Adoptium\jdk-17*") +
                 glob.glob(r"C:\Program Files\Microsoft\jdk-17*") +
                 glob.glob(r"C:\Program Files\Zulu\zulu-17*"))
    else:                                                    # Linux
        cands = (glob.glob("/usr/lib/jvm/java-17-*") +
                 glob.glob("/usr/lib/jvm/temurin-17-*"))
    for c in cands:
        if os.path.isdir(c):
            return c
    return None

_jh = _java17_home()
assert _jh, ("Java 17 not found. Install it:\n"
             "  macOS:   brew install openjdk@17\n"
             "  Windows: download Temurin 17 from https://adoptium.net\n"
             "  Linux:   sudo apt install openjdk-17-jdk")
os.environ["JAVA_HOME"] = _jh
_sep = ";" if platform.system() == "Windows" else ":"
os.environ["PATH"] = os.path.join(_jh, "bin") + _sep + os.environ["PATH"]
print("JAVA_HOME =", _jh)

os.environ["SPARK_LOCAL_IP"]        = "127.0.0.1"
os.environ["SPARK_LOCAL_HOSTNAME"]  = "localhost"
os.environ["PYSPARK_PYTHON"]        = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("US_FLIGHT_TEACHING_DASHBOARD")
    .config("spark.driver.host",            "localhost")
    .config("spark.driver.bindAddress",     "127.0.0.1")
    .config("spark.local.ip",               "127.0.0.1")
    .config("spark.driver.memory",          "4g")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.port",                "4040")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark UI:", spark.sparkContext.uiWebUrl)


JAVA_HOME = /usr/lib/jvm/java-17-openjdk-amd64
Spark UI: http://localhost:4040


In [2]:
# ==========================================================
# LOAD AND CLEAN DATA
# ----------------------------------------------------------
# Source: US Domestic Flight On-Time Performance, 2024 (BTS)
# ~7.08M rows, 37 columns, parquet.
#
# >>> EDIT THIS PATH if your file lives elsewhere <<<
# ==========================================================
FILE_PATH = r"/content/dataset.parquet"

flights_df = spark.read.parquet(FILE_PATH)

flights_clean = (
    flights_df
    .select(
        "fl_date", "op_unique_carrier", "op_carrier_fl_num",
        "origin", "origin_city_name", "origin_state_nm",
        "dest", "dest_city_name",
        "crs_dep_time", "dep_time", "dep_delay", "arr_delay",
        "is_delayed", "cancelled", "diverted", "cancellation_code",
        "distance", "air_time", "dep_time_block", "month", "day_of_month",
        "carrier_delay", "weather_delay", "nas_delay",
        "security_delay", "late_aircraft_delay",
    )
    # keep cancelled flights (dep_delay/arr_delay are null for them; averages skip nulls)
    .filter(F.col("crs_dep_time").isNotNull())
    .filter(F.col("distance") > 0)
    # decompose scheduled departure HHMM into hour / minute
    .withColumn("dep_hh", (F.col("crs_dep_time") / 100).cast("int"))
    .withColumn("dep_hh", F.when(F.col("dep_hh") >= 24, 0).otherwise(F.col("dep_hh")))
    .withColumn("dep_mm", (F.col("crs_dep_time") % 100).cast("int"))
    .withColumn("dep_mm", F.when(F.col("dep_mm") >= 60, 0).otherwise(F.col("dep_mm")))
    # REAL event time: actual calendar date + scheduled HH:MM  (used for chronological replay)
    .withColumn(
        "event_time_real",
        F.to_timestamp(F.concat_ws(" ", F.col("fl_date"),
                                   F.format_string("%02d:%02d:00", F.col("dep_hh"), F.col("dep_mm"))))
    )
    # SYNTHETIC time-of-day: every flight pinned to one day (used for the general sample mode)
    .withColumn(
        "event_time_tod",
        F.to_timestamp(F.format_string("2024-01-01 %02d:%02d:00", F.col("dep_hh"), F.col("dep_mm")))
    )
    # the five attributed causes sum to arr_delay for delayed flights (BTS convention)
    .withColumn(
        "total_cause_delay",
        F.col("carrier_delay") + F.col("weather_delay") + F.col("nas_delay")
        + F.col("security_delay") + F.col("late_aircraft_delay")
    )
    # operated = actually flew. Cancelled flights are coded is_delayed=0 in the raw
    # data, so delay RATES use operated flights as the denominator, otherwise mass
    # cancellations (e.g. CrowdStrike) silently dilute the delay percentage.
    .withColumn(
        "is_operated",
        ((F.col("cancelled") == 0) & (F.col("diverted") == 0)).cast("int")
    )
)

# ==========================================================
# ANALYSIS MODES — pick one from the dashboard dropdown
# ----------------------------------------------------------
# 1. CrowdStrike replay : Jul 17–25, 2024 in real chronological order.
#    Two calm days, then the Jul 19 crash unfolds batch by batch.
#    Many SMALL batches (50 x 500) = a fine-grained, real-life drip.
# 2. Overall sample     : random 10k sample on a time-of-day axis
#    (the original teaching stream, 10 x 1000).
# 3. Full dataset       : one-shot exact analysis of all ~7.08M rows;
#    no streaming, the charts/maps/KPIs show population truth.
# ==========================================================
MODES = {
    "Stream: CrowdStrike outage replay (Jul 17-25, chronological)":
        dict(key="crowdstrike", window="6 hours",   fmt="%b %d %H:%M",
             batch_size=500,  n_batches=50, pause=0.4),
    "Stream: overall sample (time-of-day)":
        dict(key="sample",      window="10 minutes", fmt="%H:%M",
             batch_size=1000, n_batches=10, pause=1.0),
    "Analyse full dataset (all rows, one shot)":
        dict(key="full",        window="10 minutes", fmt="%H:%M"),
}

# ---- Source 1: CrowdStrike event slice (built once, streamed chronologically) ----
CS_CARRIERS = ["DL", "AA", "UA", "WN"]   # Delta meltdown vs peers that recovered fast
CS_DAY_FROM, CS_DAY_TO = 17, 25          # Jul 17-18 calm | 19-23 outage | 24-25 recovery
CS_SAMPLE_N = 25000                      # = 50 batches x 500 rows

_cs_slice = (
    flights_clean
    .filter((F.col("month") == 7) &
            (F.col("day_of_month").between(CS_DAY_FROM, CS_DAY_TO)) &
            (F.col("op_unique_carrier").isin(CS_CARRIERS)))
    .withColumn("event_time", F.col("event_time_real"))
)
_frac = min(1.0, (CS_SAMPLE_N * 1.2) / max(_cs_slice.count(), 1))
_cs_sampled = _cs_slice.sample(withReplacement=False, fraction=_frac, seed=42).limit(CS_SAMPLE_N)
# row_id in REAL chronological order -> batches advance through time (Jul 17 -> Jul 25)
_w_cs = Window.orderBy(F.col("event_time").asc(), F.monotonically_increasing_id())
cs_source = _cs_sampled.withColumn("row_id", F.row_number().over(_w_cs) - 1).cache()

# ---- Source 2: overall random sample (time-of-day axis) ----
SAMPLE_N = 10000
_frac2 = min(1.0, (SAMPLE_N * 1.5) / 7_079_079)
_sm = (flights_clean
       .withColumn("event_time", F.col("event_time_tod"))
       .sample(withReplacement=False, fraction=_frac2, seed=42).limit(SAMPLE_N))
_w_sm = Window.orderBy(F.monotonically_increasing_id())
sample_source = _sm.withColumn("row_id", F.row_number().over(_w_sm) - 1).cache()

# ---- Source 3: the whole dataset (for the one-shot full analysis) ----
full_source = flights_clean.withColumn("event_time", F.col("event_time_tod"))

print(f"CrowdStrike replay source : {cs_source.count():,} rows "
      f"(chronological, Jul {CS_DAY_FROM}-{CS_DAY_TO}, carriers {CS_CARRIERS})")
print(f"Overall sample source     : {sample_source.count():,} rows (time-of-day)")
print(f"Full dataset source       : ~7.08M rows (aggregated on demand)")
_rng = cs_source.select(F.min('event_time').alias('a'), F.max('event_time').alias('b')).first()
print(f"Replay time span          : {_rng['a']}  ->  {_rng['b']}")


CrowdStrike replay source : 25,000 rows (chronological, Jul 17-25, carriers ['DL', 'AA', 'UA', 'WN'])
Overall sample source     : 10,000 rows (time-of-day)
Full dataset source       : ~7.08M rows (aggregated on demand)
Replay time span          : 2024-07-17 00:13:00  ->  2024-07-25 23:59:00


In [3]:
AIRPORT_COORDS = {
    "ABE":(40.6521,-75.4408,"Allentown"), "ABI":(32.4113,-99.6819,"Abilene"),
    "ABQ":(35.0402,-106.6092,"Albuquerque"), "ABR":(45.4491,-98.4218,"Aberdeen SD"),
    "ABY":(31.5355,-84.1945,"Albany GA"), "ALB":(42.7483,-73.8017,"Albany"),
    "AMA":(35.2194,-101.7059,"Amarillo"), "ANC":(61.1743,-149.9962,"Anchorage"),
    "ASE":(39.2232,-106.8688,"Aspen"), "ATL":(33.6407,-84.4277,"Atlanta"),
    "ATW":(44.2581,-88.5191,"Appleton"), "AUS":(30.1975,-97.6664,"Austin"),
    "AVL":(35.4362,-82.5418,"Asheville"), "AVP":(41.3385,-75.7234,"Scranton"),
    "BDL":(41.9389,-72.6832,"Hartford"), "BGR":(44.8074,-68.8281,"Bangor"),
    "BHM":(33.5629,-86.7535,"Birmingham"), "BIL":(45.8077,-108.5429,"Billings"),
    "BLI":(48.7927,-122.5375,"Bellingham"), "BLV":(38.5452,-89.8351,"Belleville IL"),
    "BMI":(40.4771,-88.9159,"Bloomington IL"), "BNA":(36.1245,-86.6782,"Nashville"),
    "BOI":(43.5644,-116.2228,"Boise"), "BOS":(42.3656,-71.0096,"Boston"),
    "BTM":(45.9548,-112.4975,"Butte"), "BTR":(30.5332,-91.1496,"Baton Rouge"),
    "BTV":(44.472,-73.1533,"Burlington VT"), "BUF":(42.9405,-78.7322,"Buffalo"),
    "BUR":(34.2007,-118.3585,"Burbank"), "BWI":(39.1774,-76.6684,"Baltimore"),
    "BZN":(45.7775,-111.153,"Bozeman"), "CAE":(33.9388,-81.1195,"Columbia SC"),
    "CAK":(40.9161,-81.4422,"Akron-Canton"), "CHA":(35.0353,-85.2038,"Chattanooga"),
    "CHS":(32.8986,-80.0405,"Charleston"), "CID":(41.8847,-91.7108,"Cedar Rapids"),
    "CLE":(41.4117,-81.8498,"Cleveland"), "CLT":(35.214,-80.9431,"Charlotte"),
    "CMH":(39.998,-82.8919,"Columbus"), "COS":(38.8058,-104.7008,"Colorado Springs"),
    "CRP":(27.7704,-97.5012,"Corpus Christi"), "CRW":(38.3731,-81.5932,"Charleston WV"),
    "CVG":(39.0489,-84.6678,"Cincinnati"), "DAB":(29.1799,-81.0581,"Daytona Beach"),
    "DAL":(32.8471,-96.8518,"Dallas Love"), "DAY":(39.9024,-84.2194,"Dayton"),
    "DCA":(38.8512,-77.0402,"Washington Reagan"), "DEN":(39.8561,-104.6737,"Denver"),
    "DFW":(32.8998,-97.0403,"Dallas-Fort Worth"), "DSM":(41.534,-93.6631,"Des Moines"),
    "DTW":(42.2162,-83.3554,"Detroit"), "ECP":(30.3417,-85.7973,"Panama City FL"),
    "EGE":(39.6426,-106.9177,"Vail/Eagle"), "ELM":(42.1599,-76.8916,"Elmira"),
    "ELP":(31.8072,-106.3781,"El Paso"), "ERI":(42.0831,-80.1739,"Erie"),
    "EUG":(44.1246,-123.212,"Eugene"), "EVV":(38.037,-87.5324,"Evansville"),
    "EWR":(40.6895,-74.1745,"Newark"), "EYW":(24.5561,-81.7596,"Key West"),
    "FAR":(46.9207,-96.8158,"Fargo"), "FAT":(36.7762,-119.7181,"Fresno"),
    "FCA":(48.3105,-114.256,"Kalispell"), "FLL":(26.0742,-80.1506,"Fort Lauderdale"),
    "FNT":(42.9655,-83.7436,"Flint"), "FSD":(43.582,-96.7419,"Sioux Falls"),
    "FWA":(40.9785,-85.1951,"Fort Wayne"), "GEG":(47.6199,-117.5339,"Spokane"),
    "GNV":(29.6901,-82.2718,"Gainesville"), "GPT":(30.4073,-89.0701,"Gulfport"),
    "GRB":(44.4851,-88.1296,"Green Bay"), "GRK":(31.0672,-97.8289,"Killeen"),
    "GRR":(42.8808,-85.5228,"Grand Rapids"), "GSO":(36.0978,-79.9373,"Greensboro"),
    "GSP":(34.8957,-82.2189,"Greenville"), "GTF":(47.482,-111.3707,"Great Falls"),
    "HDN":(40.4812,-107.2177,"Hayden CO"), "HLN":(46.6068,-111.9827,"Helena"),
    "HNL":(21.3187,-157.9225,"Honolulu"), "HOU":(29.6454,-95.2789,"Houston Hobby"),
    "HPN":(41.067,-73.7076,"Westchester"), "HRL":(26.2285,-97.6544,"Harlingen"),
    "HSV":(34.6372,-86.7751,"Huntsville"), "IAD":(38.9531,-77.4565,"Washington Dulles"),
    "IAH":(29.9902,-95.3368,"Houston Bush"), "ICT":(37.6499,-97.4331,"Wichita"),
    "IDA":(43.5146,-112.0708,"Idaho Falls"), "ILM":(34.2706,-77.9026,"Wilmington NC"),
    "IND":(39.7173,-86.2944,"Indianapolis"), "ISP":(40.7952,-73.1002,"Long Island"),
    "ITO":(19.7214,-155.0485,"Hilo"), "JAC":(43.6073,-110.7377,"Jackson Hole"),
    "JAN":(32.3112,-90.0759,"Jackson MS"), "JAX":(30.4941,-81.6879,"Jacksonville"),
    "JFK":(40.6413,-73.7781,"New York JFK"), "KOA":(19.7388,-156.0456,"Kona"),
    "LAS":(36.084,-115.1537,"Las Vegas"), "LAX":(33.9416,-118.4085,"Los Angeles"),
    "LBB":(33.6636,-101.8228,"Lubbock"), "LEX":(38.0365,-84.6059,"Lexington"),
    "LFT":(30.2053,-91.9876,"Lafayette"), "LGA":(40.7769,-73.874,"New York LaGuardia"),
    "LGB":(33.8177,-118.1516,"Long Beach"), "LIH":(21.976,-159.339,"Lihue"),
    "LIT":(34.7294,-92.2243,"Little Rock"), "MAF":(31.9425,-102.2019,"Midland"),
    "MCI":(39.2976,-94.7139,"Kansas City"), "MCO":(28.4312,-81.3081,"Orlando"),
    "MDW":(41.7868,-87.7522,"Chicago Midway"), "MEM":(35.0424,-89.9767,"Memphis"),
    "MFR":(42.3742,-122.8735,"Medford"), "MHT":(42.9326,-71.4357,"Manchester NH"),
    "MIA":(25.7959,-80.287,"Miami"), "MKE":(42.9472,-87.8966,"Milwaukee"),
    "MLB":(28.1028,-80.6453,"Melbourne"), "MLI":(41.4485,-90.5075,"Moline"),
    "MOB":(30.6912,-88.2428,"Mobile"), "MRY":(36.587,-121.8429,"Monterey"),
    "MSN":(43.1399,-89.3375,"Madison"), "MSO":(46.9163,-114.0906,"Missoula"),
    "MSP":(44.8848,-93.2223,"Minneapolis"), "MSY":(29.9934,-90.2581,"New Orleans"),
    "MTJ":(38.5098,-107.894,"Montrose"), "MYR":(33.6797,-78.9283,"Myrtle Beach"),
    "OAK":(37.7126,-122.2197,"Oakland"), "OGG":(20.8986,-156.4305,"Maui"),
    "OGS":(44.6819,-75.4655,"Ogdensburg"), "OKC":(35.3931,-97.6007,"Oklahoma City"),
    "OMA":(41.3032,-95.8941,"Omaha"), "ONT":(34.056,-117.6012,"Ontario CA"),
    "ORD":(41.9742,-87.9073,"Chicago O'Hare"), "ORF":(36.8946,-76.2012,"Norfolk"),
    "PBI":(26.6832,-80.0956,"West Palm Beach"), "PDX":(45.5887,-122.5975,"Portland OR"),
    "PGD":(26.9202,-81.9905,"Punta Gorda"), "PHL":(39.8744,-75.2424,"Philadelphia"),
    "PHX":(33.4342,-112.0116,"Phoenix"), "PIA":(40.6642,-89.6933,"Peoria"),
    "PIE":(27.9102,-82.6874,"St. Petersburg"), "PIT":(40.4915,-80.2329,"Pittsburgh"),
    "PNS":(30.4734,-87.1866,"Pensacola"), "PSC":(46.2647,-119.119,"Pasco WA"),
    "PSP":(33.8297,-116.5067,"Palm Springs"), "PVD":(41.724,-71.4282,"Providence"),
    "PVU":(40.2192,-111.7233,"Provo"), "PWM":(43.6462,-70.3093,"Portland ME"),
    "RAP":(44.0453,-103.0574,"Rapid City"), "RDM":(44.2541,-121.15,"Bend OR"),
    "RDU":(35.8801,-78.788,"Raleigh-Durham"), "RIC":(37.5052,-77.3197,"Richmond"),
    "RNO":(39.4991,-119.7681,"Reno"), "ROA":(37.3255,-79.9754,"Roanoke"),
    "ROC":(43.1189,-77.6724,"Rochester"), "RSW":(26.5362,-81.7552,"Fort Myers"),
    "SAN":(32.7338,-117.1933,"San Diego"), "SAT":(29.5337,-98.4698,"San Antonio"),
    "SAV":(32.1276,-81.2021,"Savannah"), "SBA":(34.4262,-119.8404,"Santa Barbara"),
    "SBN":(41.7087,-86.3173,"South Bend"), "SBP":(35.2368,-120.6424,"San Luis Obispo"),
    "SDF":(38.1744,-85.736,"Louisville"), "SEA":(47.4502,-122.3088,"Seattle"),
    "SFB":(28.7776,-81.2375,"Sanford"), "SFO":(37.6213,-122.379,"San Francisco"),
    "SGF":(37.2457,-93.3886,"Springfield MO"), "SHV":(32.4466,-93.8256,"Shreveport"),
    "SJC":(37.3639,-121.9289,"San Jose"), "SJU":(18.4394,-66.0018,"San Juan PR"),
    "SLC":(40.7899,-111.9791,"Salt Lake City"), "SMF":(38.6951,-121.5908,"Sacramento"),
    "SNA":(33.6757,-117.8682,"Orange County"), "SRQ":(27.3954,-82.5544,"Sarasota"),
    "STL":(38.7487,-90.37,"St. Louis"), "STS":(38.509,-122.8128,"Santa Rosa"),
    "STT":(18.3373,-64.9734,"St. Thomas"), "SYR":(43.1112,-76.1063,"Syracuse"),
    "TLH":(30.3965,-84.3503,"Tallahassee"), "TPA":(27.9755,-82.5332,"Tampa"),
    "TRI":(36.4752,-82.4074,"Tri-Cities TN"), "TUL":(36.1984,-95.8881,"Tulsa"),
    "TUS":(32.1161,-110.941,"Tucson"), "TYS":(35.811,-83.994,"Knoxville"),
    "VPS":(30.4832,-86.5254,"Destin-Ft Walton"), "XNA":(36.2819,-94.3068,"Fayetteville AR"),
    "YKM":(46.5682,-120.544,"Yakima"),
}


In [4]:
# ==========================================================
# TEACHING CONTENT
# Every explanation is grounded in the actual dataset:
# US Domestic Flight On-Time Performance, 2024 (BTS),
# ~7.08 million rows, 37 columns. Streaming is simulated by
# replaying a representative 10,000-row sample in batches.
# ==========================================================
TEACHING_CONTENT = {

    # ============================ VOLUME ============================
    "Volume: Flight count over time": {
        "title": "Tumbling windows on a departure timestamp",
        "dataset_context": (
            "The dataset gives fl_date (a calendar day) and crs_dep_time as an integer in HHMM "
            "form, e.g. 1209 means 12:09. We build event_time from these. In CrowdStrike mode "
            "event_time is the real calendar timestamp (fl_date + HH:MM), so the stream replays "
            "July 17-25 2024 in true chronological order and you watch the outage arrive. In the "
            "general sample mode every flight is instead pinned to one synthetic day so the "
            "windows show the average daily departure rhythm."
        ),
        "concept": (
            "A tumbling window slices that 24-hour clock into fixed 10-minute buckets and "
            "counts how many flights are scheduled to depart inside each bucket. "
            "Every flight belongs to exactly one bucket, with no overlap. "
            "The result is the national departure curve across a day."
        ),
        "how_it_works": (
            "The code runs groupBy(F.window('event_time','10 minutes')).count(). "
            "Spark rounds each scheduled time down to the nearest 10-minute boundary and "
            "groups by it. Each bar in Chart A is one bucket. As more batches stream in, "
            "the bars grow because more flights land in each window."
        ),
        "key_points": [
            "Pinning every flight to one synthetic day turns absolute datetime into a "
            "time-of-day axis. That is a deliberate modelling choice so windows show the "
            "daily departure rhythm rather than scattering across 365 dates.",
            "In a real Kafka pipeline, event_time would be the scheduled or actual departure "
            "stamped on each message, not the time Kafka received it (event vs ingestion time).",
            "The same window call runs on a live stream by swapping spark.read.parquet for "
            "spark.readStream.format('kafka'). The aggregation logic is identical.",
            "Tumbling windows never overlap: a 12:09 departure goes only into the 12:00 bucket. "
            "A sliding window would place it into several overlapping buckets.",
            "Window width is a tuning knob. Ten minutes is fine here; one-minute windows would "
            "look noisy on a 10,000-row sample.",
        ],
        "references": [
            ("BTS On-Time Performance (data source)",
             "https://www.transtats.bts.gov/Tables.asp?DB_ID=120"),
            ("Spark Structured Streaming: event-time windows",
             "https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html#window-operations-on-event-time"),
            ("Kafka: event time vs processing time",
             "https://kafka.apache.org/documentation/#design_pull"),
        ],
    },
    "Volume: Active airports over time": {
        "title": "countDistinct(origin) per window",
        "dataset_context": (
            "Each row carries an origin airport code (e.g. ATL, DFW, ORD) plus origin_city_name "
            "and origin_state_nm. Across the full dataset there are several hundred distinct "
            "origin airports; your sample typically sees ~300."
        ),
        "concept": (
            "Counting rows tells you volume; counting distinct origins tells you breadth. "
            "Chart B plots countDistinct(origin) per 10-minute window: how many different "
            "airports are pushing departures into the network at that time of day."
        ),
        "how_it_works": (
            "The aggregation adds countDistinct('origin') alongside the flight count. "
            "Early-morning windows light up many regional airports at once; late-night windows "
            "shrink to a handful of 24-hour hubs."
        ),
        "key_points": [
            "countDistinct is a wide aggregation: Spark must deduplicate across the whole window, "
            "which is heavier than a plain count and shuffles more data.",
            "On true streams an exact distinct count is expensive, so production systems often use "
            "approx_count_distinct (HyperLogLog) to trade a little accuracy for speed.",
            "Breadth and volume diverge: a few mega-hubs can dominate volume while distinct-airport "
            "count stays flat, which is itself a signal about network shape.",
        ],
        "references": [
            ("Spark SQL: approx_count_distinct",
             "https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.approx_count_distinct.html"),
            ("HyperLogLog explained",
             "https://en.wikipedia.org/wiki/HyperLogLog"),
        ],
    },
    "Volume: Departure rush pattern": {
        "title": "Time-of-day demand via dep_time_block",
        "dataset_context": (
            "The dataset pre-bins each flight into dep_time_block: Morning, Afternoon, Evening, "
            "or Overnight. This is a ready-made categorical you can group on directly without "
            "parsing crs_dep_time yourself."
        ),
        "concept": (
            "Departures are not uniform across the day. Chart A breaks the current batch into the "
            "four dep_time_block categories so you can see the morning and evening peaks that drive "
            "airport staffing, gate pressure, and downstream delay propagation."
        ),
        "how_it_works": (
            "Chart A groups the batch by dep_time_block and counts each category. The window bars "
            "in the other Volume panels show the same demand at finer 10-minute resolution; here it "
            "is collapsed into four interpretable blocks."
        ),
        "key_points": [
            "Pre-binned categories like dep_time_block are cheap to group on but lose resolution; "
            "the 10-minute window view keeps detail. Pick the grain that matches the question.",
            "Evening peaks matter because late-aircraft delay (a real column here) propagates: a "
            "plane late in the morning is often the same tail number late again that evening.",
            "Overnight has the fewest departures but a different delay profile, dominated by red-eye "
            "and freight-adjacent operations.",
        ],
        "references": [
            ("BTS reporting-carrier on-time documentation",
             "https://www.transtats.bts.gov/Fields.asp?gnoyr_VQ=FGJ"),
            ("Spark groupBy aggregations",
             "https://spark.apache.org/docs/latest/sql-getting-started.html#aggregations"),
        ],
    },

    # ============================ DELAY ============================
    "Delay: Average departure delay": {
        "title": "avg(dep_delay) per tumbling window",
        "dataset_context": (
            "dep_delay is the scheduled-to-actual departure gap in minutes. Negative means the "
            "flight left early; positive means late. It is null for cancelled flights, which Spark "
            "averages skip automatically."
        ),
        "concept": (
            "Averaging dep_delay inside each 10-minute window turns millions of individual gaps "
            "into a readable delay curve across the day. Chart A plots avg(dep_delay) per window as "
            "a line with a shaded area."
        ),
        "how_it_works": (
            "The aggregation computes round(avg('dep_delay'),2) per window. Early-morning windows "
            "usually sit near zero or negative (the system is fresh); the average climbs through the "
            "day as congestion and late-aircraft knock-on accumulate."
        ),
        "key_points": [
            "Averages hide spread. A window can average +5 minutes while containing both a 90-minute "
            "delay and many on-time flights. The histogram panels expose that distribution.",
            "dep_delay null for cancellations is why the cancellation count is tracked separately; "
            "a cancelled flight is not a zero-minute delay, it is an absence.",
            "On a real stream you would add a watermark so late-arriving events past a threshold are "
            "dropped rather than reopening old windows indefinitely.",
        ],
        "references": [
            ("Spark Structured Streaming: watermarking",
             "https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html#handling-late-data-and-watermarking"),
            ("BTS On-Time Performance glossary",
             "https://www.transtats.bts.gov/Fields.asp?gnoyr_VQ=FGJ"),
        ],
    },
    "Delay: Departure vs arrival delay": {
        "title": "dep_delay against arr_delay: in-air recovery",
        "dataset_context": (
            "Two delay columns exist: dep_delay at the gate and arr_delay at the destination. "
            "The difference reflects what happened in between, captured by air_time, taxi_out, and "
            "taxi_in. Crews routinely make up time in the air."
        ),
        "concept": (
            "Chart B overlays avg(arr_delay) on avg(dep_delay) per window. When the arrival line "
            "sits below the departure bars, flights are recovering time en route; when it sits "
            "above, problems compounded after pushback."
        ),
        "how_it_works": (
            "Departure delay is drawn as bars and arrival delay as a line on a second axis, so the "
            "gap between them is the average in-air recovery (or loss) for that window."
        ),
        "key_points": [
            "Padded schedules mean crs_elapsed_time is often longer than needed, so modest departure "
            "delays vanish by arrival. This is deliberate airline scheduling, not luck.",
            "Weather and NAS (air-traffic) delays tend to grow the dep-to-arr gap; carrier delays "
            "are more often absorbed.",
            "Plotting two related series on twin axes is a teaching pattern: it shows correlation "
            "without forcing the eye to compute the difference manually.",
        ],
        "references": [
            ("FAA: causes of flight delays",
             "https://www.faa.gov/nextgen/programs/weather/faq"),
            ("Matplotlib twin axes",
             "https://matplotlib.org/stable/gallery/subplots_axes_and_figures/two_scales.html"),
        ],
    },
    "Delay: Delay causes breakdown": {
        "title": "Five cause columns: where the minutes go",
        "dataset_context": (
            "When a flight is delayed, BTS attributes the minutes across five columns: "
            "carrier_delay, weather_delay, nas_delay, security_delay, and late_aircraft_delay. "
            "For on-time flights all five are zero, so they sum to the reportable delay."
        ),
        "concept": (
            "Chart B sums each cause across the current batch and ranks them. This answers the "
            "question a delay average cannot: not how late, but why. Late-aircraft and carrier "
            "causes usually dominate; security is almost always negligible."
        ),
        "how_it_works": (
            "The batch is reduced with sum() over the five cause columns, then drawn as a ranked "
            "bar chart. Because the columns are pre-attributed by BTS, no inference is needed."
        ),
        "key_points": [
            "late_aircraft_delay is contagion: a late inbound tail number delays its next leg. It is "
            "often the single largest bucket and explains why delay builds through the day.",
            "nas_delay covers the National Airspace System, which includes air-traffic control flow "
            "control and non-extreme weather routing.",
            "These columns are the taxi dataset's missing dimension: fares had no causal attribution, "
            "but every delay here is decomposed into reasons out of the box.",
        ],
        "references": [
            ("BTS: understanding the reporting of causes of flight delays",
             "https://www.bts.gov/topics/airlines-and-airports/understanding-reporting-causes-flight-delays-and-cancellations"),
            ("Spark aggregate functions",
             "https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html"),
        ],
    },
    "Delay: Delay vs Distance": {
        "title": "Does route length predict delay?",
        "dataset_context": (
            "distance is the great-circle route length in miles; it ranges from short hops under "
            "100 miles to transcontinental routes over 2,500. air_time is the actual minutes aloft."
        ),
        "concept": (
            "Chart B samples the batch and plots distance on X against dep_delay on Y, coloured by "
            "air_time. It tests a common intuition: are long-haul flights more or less prone to "
            "departure delay than short hops? The cloud usually shows weak correlation."
        ),
        "how_it_works": (
            "Up to 400 flights are sampled to keep the scatter legible, then drawn as dots. "
            "Distance and air_time are tightly linked, so the colour gradient tracks the X axis."
        ),
        "key_points": [
            "Departure delay is set at the gate, before distance matters, so the scatter is mostly "
            "vertical noise: distance is a weak predictor of leaving late.",
            "Distance matters more for arrival delay, because longer flights have more room to "
            "recover time in the air.",
            "Sampling before plotting is essential at scale: rendering millions of points would "
            "freeze the browser and hide structure under overdraw.",
        ],
        "references": [
            ("Great-circle distance",
             "https://en.wikipedia.org/wiki/Great-circle_distance"),
            ("Sampling for visualization",
             "https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.sample.html"),
        ],
    },

    # ============================ CARRIER ============================
    "Carrier: Flight share": {
        "title": "Flights per op_unique_carrier",
        "dataset_context": (
            "op_unique_carrier is the airline code: AA (American), DL (Delta), UA (United), "
            "WN (Southwest), plus regionals like OO, 9E, MQ and ultra-low-cost carriers like G4 "
            "(Allegiant). Regionals fly many short legs, so flight share is not the same as seat share."
        ),
        "concept": (
            "Chart A counts flights per carrier in the current batch and ranks them. This is market "
            "structure by operation count, the carrier analogue of the taxi dataset's passenger-count "
            "distribution."
        ),
        "how_it_works": (
            "The batch is grouped by op_unique_carrier with value_counts, top 12 carriers shown as "
            "bars. As batches accumulate the ranking stabilises toward the national mix."
        ),
        "key_points": [
            "Operation count over-weights regional carriers that fly many short legs under mainline "
            "brands, so a high bar does not mean a large airline by passengers.",
            "Carrier is a clean categorical key, ideal for groupBy and for joining external fleet or "
            "on-time reputation tables.",
            "Pairing this with the on-time and delay-by-carrier panels turns raw share into a "
            "volume-versus-reliability story.",
        ],
        "references": [
            ("BTS carrier codes",
             "https://www.transtats.bts.gov/Fields.asp?gnoyr_VQ=FGJ"),
            ("Spark value counts pattern",
             "https://spark.apache.org/docs/latest/sql-getting-started.html#aggregations"),
        ],
    },
    "Carrier: On-time performance": {
        "title": "On-time rate per carrier from is_delayed",
        "dataset_context": (
            "is_delayed is a 0/1 flag the dataset provides per flight. Averaging it gives the delay "
            "rate directly; one minus that is the on-time rate. The industry convention counts a "
            "flight delayed at 15+ minutes."
        ),
        "concept": (
            "Chart B computes the on-time percentage for each carrier so you can rank reliability "
            "rather than volume. A high-volume carrier with a poor on-time rate is a very different "
            "operation from a small one with a strong record."
        ),
        "how_it_works": (
            "The batch is grouped by carrier and avg(is_delayed) is converted to an on-time "
            "percentage. Bars are ordered so the most punctual carrier in the sample stands out."
        ),
        "key_points": [
            "A 0/1 flag averaged becomes a rate. This is the cheapest possible way to compute a "
            "proportion in SQL and streams cleanly window by window.",
            "Sample size matters: a regional with few flights in one batch can show a misleading "
            "rate, which is why the figure steadies as batches accumulate.",
            "On-time rate and average delay can disagree: one very late flight barely moves the rate "
            "but lifts the average, so report both.",
        ],
        "references": [
            ("BTS on-time definition (15-minute threshold)",
             "https://www.bts.gov/topics/airlines-and-airports/airline-time-performance-and-causes-flight-delays"),
            ("Spark avg on boolean/int flags",
             "https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.avg.html"),
        ],
    },
    "Carrier: Delay by carrier": {
        "title": "avg(dep_delay) per carrier",
        "dataset_context": (
            "Combining op_unique_carrier with dep_delay shows which airlines run hot. Network "
            "carriers with complex hub banks behave differently from point-to-point low-cost "
            "operators."
        ),
        "concept": (
            "Chart B averages departure delay within each carrier and ranks them, exposing the "
            "operational tail: a few carriers typically carry most of the average delay while the "
            "median carrier sits close to on-time."
        ),
        "how_it_works": (
            "groupBy('op_unique_carrier').agg(avg('dep_delay')) on the batch, drawn as ranked bars "
            "for the same carriers shown in the share panel for a direct comparison."
        ),
        "key_points": [
            "Hub-and-spoke carriers concentrate delay in connecting banks; the average can mask that "
            "structure, which a per-hub map view recovers.",
            "Late-aircraft propagation means a carrier's morning performance partly determines its "
            "evening average, so delay is autocorrelated within an airline's day.",
            "Ranking carriers is a groupBy plus order-by, the workhorse pattern of batch analytics "
            "and a clean teaching example of reduce-then-sort.",
        ],
        "references": [
            ("BTS airline on-time statistics",
             "https://www.bts.gov/topics/airlines-and-airports/airline-time-performance-and-causes-flight-delays"),
            ("Spark groupBy + orderBy",
             "https://spark.apache.org/docs/latest/sql-getting-started.html#aggregations"),
        ],
    },

    # ============================ SPATIAL ============================
    "Spatial: Airport volume heatmap": {
        "title": "Origin volume on a US map via an embedded coordinate lookup",
        "dataset_context": (
            "This dataset has no GPS coordinates, only airport codes and city/state names. To map "
            "anything, each origin code is joined to an embedded lookup of ~185 major US airports "
            "(lat, lon), which covers about 97% of all flights in the data."
        ),
        "concept": (
            "Chart A and Map 1 aggregate accumulated flights by origin, attach coordinates, and "
            "render a heatmap weighted by flight volume. The familiar national pattern emerges: dense "
            "in the Northeast corridor, Texas, California, and the Atlanta/Chicago hubs."
        ),
        "how_it_works": (
            "Flights are grouped by origin in Spark, the small per-airport result is pulled to "
            "pandas, coordinates are mapped in, and folium HeatMap weights each point by its count. "
            "Only the aggregated airports cross to the driver, never the raw rows."
        ),
        "key_points": [
            "The map is built from an aggregate, not raw rows: a few hundred airport points instead "
            "of millions of flights. Aggregate-then-collect is the rule for mapping big data.",
            "Coverage is finite. Flights from airports outside the 185-entry lookup (about 3%, mostly "
            "small regionals and territories) still count in the charts but cannot be placed on the map.",
            "Replacing per-flight GPS with per-airport coordinates is the central adaptation from the "
            "original taxi dashboard, which had true pickup latitude and longitude on every row.",
        ],
        "references": [
            ("folium HeatMap plugin",
             "https://python-visualization.github.io/folium/latest/user_guide/plugins/heatmap.html"),
            ("BTS master coordinate / airport reference",
             "https://www.transtats.bts.gov/Tables.asp?DB_ID=595"),
        ],
    },
    "Spatial: Worst-delay airports": {
        "title": "Per-airport avg(dep_delay) dot map",
        "dataset_context": (
            "Grouping accumulated flights by origin and averaging dep_delay ranks airports by how "
            "late departures run. Congested or weather-exposed hubs surface here even when their raw "
            "volume is moderate."
        ),
        "concept": (
            "Map 2 places one dot per airport, coloured and sized by average departure delay, so the "
            "geography of delay is visible. Chart A lists the worst offenders in the sample as a bar "
            "ranking."
        ),
        "how_it_works": (
            "Per-airport avg(dep_delay) is computed in Spark, joined to coordinates, and drawn as "
            "graduated circle markers (yellow to red) on a light basemap. Hover a dot for the city "
            "and minutes."
        ),
        "key_points": [
            "Average delay per airport is volume-robust in a way a raw heatmap is not: a quiet airport "
            "with chronic delay still lights up red.",
            "Small airports with few sampled flights can show extreme averages by chance, so read the "
            "dot size and the accumulating sample together.",
            "This is a groupBy-join-plot pipeline: reduce in Spark, enrich with a lookup, visualise in "
            "folium. The same shape powers most geospatial dashboards.",
        ],
        "references": [
            ("folium CircleMarker",
             "https://python-visualization.github.io/folium/latest/user_guide/vector_layers/circle_and_circle_marker.html"),
            ("Matplotlib colormap normalization",
             "https://matplotlib.org/stable/users/explain/colors/colormapnorms.html"),
        ],
    },
    "Spatial: Busiest airports dot map": {
        "title": "Per-airport volume dot map",
        "dataset_context": (
            "The same per-airport aggregation, but sized and coloured by flight count rather than "
            "delay. The hubs (ATL, DFW, DEN, ORD, LAX) dominate, matching published US traffic "
            "rankings."
        ),
        "concept": (
            "Map 2 sizes each dot by departure volume, giving a cleaner read of the busiest origins "
            "than an overlapping heatmap. Chart A shows the same top airports as a horizontal bar "
            "ranking."
        ),
        "how_it_works": (
            "Per-airport counts are normalised between the 5th and 95th percentiles so a few mega-hubs "
            "do not flatten everything else, then mapped to dot radius and colour."
        ),
        "key_points": [
            "Percentile-based normalisation keeps outliers from dominating the scale, a standard trick "
            "when one or two values are far larger than the rest.",
            "Dot maps beat heatmaps when you want to identify discrete places; heatmaps win for "
            "continuous density. The dashboard offers both deliberately.",
            "Volume and delay rankings differ: the busiest airport is not always the most delayed, "
            "which is the contrast the two spatial panels are designed to show.",
        ],
        "references": [
            ("FAA: busiest US airports",
             "https://www.faa.gov/airports/planning_capacity/passenger_allcargo_stats/passenger"),
            ("folium quickstart",
             "https://python-visualization.github.io/folium/latest/getting_started.html"),
        ],
    },
    "Spatial: Heatmap + delay dots": {
        "title": "Volume heat plus delay dots in one view",
        "dataset_context": (
            "Combines two per-airport signals: a heat layer weighted by flight volume and circle "
            "markers coloured by average delay. One layer is where the flights are, the other is "
            "where the pain is."
        ),
        "concept": (
            "Map 2 overlays both so you can spot airports that are simultaneously high-volume and "
            "high-delay, the genuine network pressure points, versus busy-but-smooth or quiet-but-"
            "troubled airports."
        ),
        "how_it_works": (
            "The heatmap is drawn first from per-airport counts, then delay-coloured circle markers "
            "are layered on top with a legend distinguishing the two encodings."
        ),
        "key_points": [
            "Layering two encodings (size/heat for one variable, colour for another) doubles the "
            "information per pixel but needs a clear legend or it confuses more than it informs.",
            "A red dot sitting on a hot patch is the worst case: heavy traffic and heavy delay "
            "together, where interventions pay off most.",
            "Both layers come from the same single groupBy on origin, so the extra view costs almost "
            "nothing beyond the render.",
        ],
        "references": [
            ("folium FeatureGroup and layered maps",
             "https://python-visualization.github.io/folium/latest/user_guide/ui_elements/feature_group.html"),
            ("Visual encoding principles",
             "https://en.wikipedia.org/wiki/Visual_variable"),
        ],
    },

    # ============================ ANOMALY ============================
    "Anomaly: Unusual demand windows": {
        "title": "Flagging windows above mean + 1.5 sigma",
        "dataset_context": (
            "Using the same 10-minute flight counts, this panel asks which windows are statistically "
            "busy relative to the day's own rhythm, rather than busy in absolute terms."
        ),
        "concept": (
            "Chart A computes the mean and standard deviation of flight_count across all windows in "
            "the sample, then colours red any window exceeding mean + 1.5 standard deviations. Those "
            "are the demand spikes."
        ),
        "how_it_works": (
            "A dashed threshold line marks mean + 1.5 sigma; bars above it turn red. The threshold "
            "moves as more batches arrive and the distribution fills in, which is exactly how a "
            "streaming anomaly rule recalibrates."
        ),
        "key_points": [
            "A mean-plus-k-sigma rule is the simplest anomaly detector and assumes roughly normal "
            "counts. It is interpretable but fooled by skew and by the rush-hour structure itself.",
            "Because the threshold is computed from the data in view, it is adaptive: on a real "
            "stream you would maintain running mean and variance instead of recomputing each batch.",
            "1.5 sigma is a sensitivity dial. Lower it and everything looks anomalous; raise it and "
            "you miss real spikes. Tuning it is the whole game.",
        ],
        "references": [
            ("Three-sigma rule",
             "https://en.wikipedia.org/wiki/68%E2%80%9395%E2%80%9399.7_rule"),
            ("Streaming anomaly detection overview",
             "https://en.wikipedia.org/wiki/Anomaly_detection"),
        ],
    },
    "Anomaly: Delay spike detection": {
        "title": "Departure-delay distribution with a spike threshold",
        "dataset_context": (
            "Instead of per-window averages, this panel looks at the raw dep_delay values in the "
            "current batch. The distribution is heavily right-skewed: most flights cluster near zero "
            "with a long tail of severe delays."
        ),
        "concept": (
            "Chart B draws a histogram of dep_delay and marks a spike threshold at mean + 2.5 "
            "standard deviations. Flights beyond it are the operationally painful outliers that "
            "averages quietly absorb."
        ),
        "how_it_works": (
            "The batch's dep_delay values are binned into a histogram and a dashed line marks the "
            "threshold. The long right tail is the visual point: delay is not symmetric."
        ),
        "key_points": [
            "Because the distribution is skewed, mean + k-sigma flags only the extreme right tail and "
            "never the early departures, which is the intended behaviour for a delay alarm.",
            "Histograms reveal shape that a single average destroys. The same +5-minute mean can come "
            "from very different distributions.",
            "On a stream you would compute the threshold from a rolling window of recent delays so the "
            "alarm tracks current conditions rather than a fixed historical cutoff.",
        ],
        "references": [
            ("Right-skewed distributions",
             "https://en.wikipedia.org/wiki/Skewness"),
            ("Matplotlib histograms",
             "https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.hist.html"),
        ],
    },
    "Anomaly: Cancellation clusters": {
        "title": "Cancellations per window and by cause code",
        "dataset_context": (
            "cancelled is a 0/1 flag and cancellation_code gives the reason: A (carrier), "
            "B (weather), C (NAS), or 'Not Cancelled'. Cancelled flights have null dep_delay, so "
            "they are tracked as counts, not as delays."
        ),
        "concept": (
            "Chart B sums cancellations per 10-minute window to reveal clusters. Cancellations are "
            "bursty: they arrive in waves tied to weather systems and air-traffic programs rather "
            "than scattering evenly."
        ),
        "how_it_works": (
            "sum('cancelled') is aggregated per window and drawn as bars; spikes mark the clustered "
            "events. The cancellation_code column lets you attribute a cluster to weather versus "
            "carrier versus NAS."
        ),
        "key_points": [
            "A cancellation is an absence, not a large delay. Treating it as a zero-minute or huge "
            "delay would corrupt the delay averages, so it is counted separately.",
            "Clustering is the signature: weather and ground-stop programs cancel many flights at "
            "once, so the bars spike rather than drift.",
            "cancellation_code turns a raw count into a cause analysis, mirroring how the five "
            "delay-cause columns decompose lateness.",
        ],
        "references": [
            ("BTS cancellation cause codes",
             "https://www.bts.gov/topics/airlines-and-airports/understanding-reporting-causes-flight-delays-and-cancellations"),
            ("FAA ground stops and ground delay programs",
             "https://www.fly.faa.gov/Information/information.html"),
        ],
    },
}


# ==========================================================
# MODE EXPLANATIONS  (for the two non-teaching-stream modes)
# Rendered in the same teaching panel; kept separate from
# TEACHING_CONTENT so they don't pollute the question dropdown.
# ==========================================================
MODE_TEACHING = {

    "__crowdstrike__": {
        "title": "Replaying the CrowdStrike outage in real chronological order",
        "dataset_context": (
            "This mode streams Delta Air Lines (DL) flights for 17\u201325 July 2024 "
            "(27,529 flights) in true chronological order. Unlike the teaching stream, "
            "which pins every flight to one synthetic day, here event_time is the REAL "
            "date-and-time built from fl_date + crs_dep_time, so the timeline advances "
            "across the nine days and the outage appears where it actually happened."
        ),
        "concept": (
            "The CrowdStrike faulty update went out on 19 July 2024 and crippled Delta's "
            "crew-tracking systems for days. Streaming the data in small 6-hour real-time "
            "slices (about 36 batches instead of the sample stream's 10) lets you watch the "
            "operation move from normal, to ignition on the 18th\u201319th, to sustained "
            "meltdown, to recovery by the 24th \u2014 the smaller-batch, higher-count design "
            "is what makes it feel like a live incident unfolding."
        ),
        "how_it_works": (
            "Each batch releases the next 6 hours of scheduled departures. Chart A tracks "
            "cancellation rate and operated-delay rate per slice with the outage window "
            "shaded; Chart B breaks delay minutes into causes so you can see carrier delay "
            "dominate. KPIs and the airport map accumulate as the incident spreads."
        ),
        "key_points": [
            "On 19 July, Delta's cancellation rate hits ~42% and ~92% of the flights that "
            "did operate arrived late \u2014 a near-total operational collapse.",
            "The smoking gun is in the causes: average carrier_delay explodes to ~67 minutes "
            "while weather_delay stays at ~0. The data itself proves this was an IT/carrier "
            "failure, not weather.",
            "Because the event is dominated by cancellations, cancellation rate and "
            "operated-delay rate are shown as SEPARATE series. Folding them into one "
            "'% delayed' would hide the meltdown, since cancelled flights are coded "
            "is_delayed=0 (the denominator problem from the data audit).",
            "Using real event_time rather than a synthetic day is what lets the spike land on "
            "the correct calendar dates instead of scattering across a single clock.",
            "Smaller batches and more of them trade render speed for temporal resolution \u2014 "
            "the closer you get to per-hour slices, the more the ramp-up and recovery detail "
            "you can see.",
        ],
        "references": [
            ("CrowdStrike outage (19 July 2024) overview",
             "https://en.wikipedia.org/wiki/2024_CrowdStrike_incident"),
            ("BTS: reporting the causes of flight delays",
             "https://www.bts.gov/topics/airlines-and-airports/understanding-reporting-causes-flight-delays-and-cancellations"),
            ("Spark event-time windows",
             "https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html#window-operations-on-event-time"),
        ],
    },

    "__overview__": {
        "title": "Whole-dataset overview \u2014 all 7.08M flights, 2024",
        "dataset_context": (
            "This mode is NOT a stream. It aggregates the entire 2024 dataset at once to give "
            "the big picture, using the population conventions from the data audit: delay rate "
            "is measured among OPERATED flights (cancelled and diverted removed), while "
            "cancellation rate is measured against all scheduled flights."
        ),
        "concept": (
            "Four whole-year views: delay rate by carrier, month-by-month seasonality of delays "
            "and cancellations, the national breakdown of delay minutes by cause, and the "
            "airports carrying the most departure delay. Together they frame everything the "
            "streaming modes then show in motion."
        ),
        "how_it_works": (
            "The panels are computed with groupBy aggregations over the full DataFrame "
            "(delay metrics over the operated-only subset), pulled to pandas once and cached, "
            "then drawn as static charts and a national airport map."
        ),
        "key_points": [
            "July is the worst month for delays (summer storms plus the CrowdStrike week), "
            "while January leads on cancellations (winter weather) \u2014 two different "
            "disruption mechanisms with different signatures.",
            "Across the year, late-aircraft and carrier causes dominate total delay minutes; "
            "weather is a smaller share of minutes than its reputation suggests, though it "
            "drives cancellations rather than in-flight delay.",
            "Delay rates here use the operated-flight denominator, so they read higher \u2014 and "
            "more honestly \u2014 than a naive delayed/all-scheduled ratio that dilutes the rate "
            "with cancellations.",
            "This static overview and the live streams share the same underlying columns and "
            "conventions, so the numbers reconcile across the whole dashboard.",
        ],
        "references": [
            ("BTS On-Time Performance (data source)",
             "https://www.transtats.bts.gov/Tables.asp?DB_ID=120"),
            ("BTS: airline on-time performance and causes of delay",
             "https://www.bts.gov/topics/airlines-and-airports/airline-time-performance-and-causes-flight-delays"),
        ],
    },
}


In [5]:
# ==========================================================
# QUESTION CATEGORY MAP  (first word of each label -> category)
# ==========================================================
QUESTION_MAP = {k: k.split(":")[0].strip().lower() for k in TEACHING_CONTENT}

# ==========================================================
# SHARED STYLE CONSTANTS
# ==========================================================
PANEL_BORDER = "1px solid #e0e0e0"
MAP_H        = "400px"
MAP_W_PX     = 460
CHART_H      = "300px"

# ==========================================================
# WIDGETS
# ==========================================================
question_dd = widgets.Dropdown(
    options    = list(TEACHING_CONTENT.keys()),
    value      = "Volume: Flight count over time",
    description= "",
    layout     = widgets.Layout(width="500px")
)
run_btn  = widgets.Button(description="Start stream", button_style="success",
                          layout=widgets.Layout(width="120px"))
stop_btn = widgets.Button(description="Stop", button_style="danger",
                          layout=widgets.Layout(width="80px"))
status_lbl = widgets.Label(value="Ready", layout=widgets.Layout(width="180px"))

header_html = widgets.HTML("""
<div style="background:#0b2545;color:#ffffff;padding:18px 24px;border-radius:10px;
            margin-bottom:12px;font-family:Arial,sans-serif;">
    <div style="font-size:20px;font-weight:700;letter-spacing:0.5px">
        US Flight Delay Live Analytics
    </div>
    <div style="font-size:13px;opacity:0.7;margin-top:4px">
        Streaming teaching dashboard &middot; 2024 BTS on-time performance
    </div>
</div>
""")

# ---- Mode selector (which analysis to run) ----
mode_dd = widgets.Dropdown(
    options = list(MODES.keys()),
    value   = list(MODES.keys())[0],
    layout  = widgets.Layout(width="500px")
)

MODE_BLURBS = {
    "crowdstrike": (
        "<b>CrowdStrike outage replay.</b> Real chronological order, Jul 17&ndash;25 2024, "
        "carriers DL / AA / UA / WN. Two calm days stream first, then the Jul 19 crash "
        "unfolds and recovers &mdash; 50 small batches of 500 flights, 6-hour windows. "
        "Watch <i>Anomaly: Cancellation clusters</i> and <i>Delay: Delay causes breakdown</i> "
        "(the spike is carrier_delay, not weather)."
    ),
    "sample": (
        "<b>Overall sample stream.</b> A random 10,000-flight sample of the whole of 2024 "
        "replayed in 10 batches of 1,000 on a time-of-day axis (10-minute windows). "
        "Shows the national daily rhythm."
    ),
    "full": (
        "<b>Full dataset analysis.</b> One-shot exact aggregation of all ~7.08M flights &mdash; "
        "no sampling, no streaming. Charts, maps and KPIs show population truth "
        "(takes a moment: several full scans). "
        "Convention everywhere: % delayed is over <b>operated</b> flights; cancellations are "
        "reported separately."
    ),
}
mode_desc = widgets.HTML()

def _update_mode(change=None):
    k = MODES[mode_dd.value]["key"]
    mode_desc.value = (
        f"<div style='background:#f4f9fc;border:1px solid #d5e8f2;border-radius:6px;"
        f"padding:8px 12px;font-size:12px;color:#333;line-height:1.55;margin:0 0 10px 0'>"
        f"{MODE_BLURBS[k]}</div>"
    )
    run_btn.description = "Run analysis" if k == "full" else "Start stream"

mode_dd.observe(_update_mode, names="value")
_update_mode()

mode_row = widgets.HBox(
    [
        widgets.HTML("<span style='font-size:13px;font-weight:600;line-height:32px;margin-right:6px'>Mode</span>"),
        mode_dd
    ],
    layout=widgets.Layout(align_items="center", gap="8px", margin="0 0 6px 0")
)

controls_row = widgets.HBox(
    [
        widgets.HTML("<span style='font-size:13px;font-weight:600;line-height:32px;margin-right:6px'>Question</span>"),
        question_dd, run_btn, stop_btn, status_lbl
    ],
    layout=widgets.Layout(align_items="center", gap="8px", margin="0 0 12px 0")
)

# ==========================================================
# OUTPUT PANELS
# ==========================================================
teach_out  = widgets.Output(layout={"border": PANEL_BORDER, "border_radius": "8px",
                                     "padding": "0px", "margin": "0 0 12px 0"})
kpi_out    = widgets.Output(layout={"border": PANEL_BORDER, "border_radius": "8px",
                                     "padding": "6px", "margin": "0 0 12px 0"})
chart1_out = widgets.Output(layout={"border": PANEL_BORDER, "border_radius": "8px",
                                     "padding": "4px", "height": CHART_H})
chart2_out = widgets.Output(layout={"border": PANEL_BORDER, "border_radius": "8px",
                                     "padding": "4px", "height": CHART_H})
map1_out   = widgets.Output(layout={"border": PANEL_BORDER, "border_radius": "8px",
                                     "padding": "2px", "height": MAP_H})
map2_out   = widgets.Output(layout={"border": PANEL_BORDER, "border_radius": "8px",
                                     "padding": "2px", "height": MAP_H})
table_out  = widgets.Output(layout={"border": PANEL_BORDER, "border_radius": "8px",
                                     "padding": "6px"})

map1_lbl = widgets.HTML("<p style='margin:6px 8px 2px;font-size:12px;font-weight:600;color:#444'>Airport volume heatmap</p>")
map2_lbl = widgets.HTML("<p style='margin:6px 8px 2px;font-size:12px;font-weight:600;color:#444'>Per-airport dot map</p>")

left_col = widgets.VBox(
    [
        widgets.HTML("<p style='margin:6px 8px 2px;font-size:12px;font-weight:600;color:#444'>Chart A</p>"),
        chart1_out,
        widgets.HTML("<p style='margin:10px 8px 2px;font-size:12px;font-weight:600;color:#444'>Chart B</p>"),
        chart2_out,
    ],
    layout=widgets.Layout(width="50%")
)
right_col = widgets.VBox([map1_lbl, map1_out, map2_lbl, map2_out],
                         layout=widgets.Layout(width="50%"))

display(
    widgets.VBox([
        header_html,
        mode_row,
        mode_desc,
        controls_row,
        teach_out,
        widgets.HTML("<p style='margin:0 0 4px;font-size:13px;font-weight:700;color:#0b2545'>Live KPIs</p>"),
        kpi_out,
        widgets.HTML("<p style='margin:8px 0 4px;font-size:13px;font-weight:700;color:#0b2545'>Charts and Maps</p>"),
        widgets.HBox([left_col, right_col],
                     layout=widgets.Layout(width="100%", align_items="flex-start", gap="8px")),
        widgets.HTML("<p style='margin:8px 0 4px;font-size:13px;font-weight:700;color:#0b2545'>Window aggregation table</p>"),
        table_out,
    ])
)

running = False

def stop_stream(b):
    global running
    running = False
    status_lbl.value = "Stopped"
stop_btn.on_click(stop_stream)

# ==========================================================
# TEACHING PANEL RENDERER
# ==========================================================
def _render_teaching_info(info):
    if not info:
        return
    refs_html = ""
    for label, url in info["references"]:
        refs_html += (
            f'<a href="{url}" target="_blank" style="display:inline-block;'
            f'margin:3px 6px 3px 0;padding:4px 10px;background:#eef3ff;color:#1a3a8f;'
            f'border:1px solid #c0cef5;border-radius:4px;font-size:11px;'
            f'text-decoration:none;">{label}</a>'
        )
    bullets_html = ""
    for point in info["key_points"]:
        bullets_html += (
            f'<div style="display:flex;align-items:flex-start;margin:5px 0;">'
            f'<span style="min-width:8px;height:8px;background:#1d9bd1;border-radius:50%;'
            f'margin:5px 10px 0 0;flex-shrink:0;display:block"></span>'
            f'<span style="font-size:13px;color:#333;line-height:1.6">{point}</span></div>'
        )
    html = f"""
    <div style="font-family:Arial,sans-serif;border-radius:8px;overflow:hidden;">
        <div style="background:#0b2545;padding:14px 20px;">
            <div style="font-size:15px;font-weight:700;color:#ffffff">{info["title"]}</div>
        </div>
        <div style="display:grid;grid-template-columns:1fr 1fr;gap:0;">
            <div style="padding:16px 18px;background:#f4f9fc;border-right:1px solid #e0e0e0;">
                <div style="font-size:11px;font-weight:700;color:#1d9bd1;letter-spacing:0.8px;
                            margin-bottom:6px;">WHAT THIS CONCEPT DOES</div>
                <p style="font-size:13px;color:#333;line-height:1.7;margin:0">{info["concept"]}</p>
            </div>
            <div style="padding:16px 18px;background:#fff;">
                <div style="font-size:11px;font-weight:700;color:#1d9bd1;letter-spacing:0.8px;
                            margin-bottom:6px;">WHAT THIS DASHBOARD IS DOING NOW</div>
                <p style="font-size:13px;color:#333;line-height:1.7;margin:0">{info["how_it_works"]}</p>
            </div>
        </div>
        <div style="padding:12px 18px;background:#f7fbfd;border-top:1px solid #e0e0e0;">
            <div style="font-size:11px;font-weight:700;color:#1d9bd1;letter-spacing:0.8px;
                        margin-bottom:6px;">DATASET CONTEXT</div>
            <p style="font-size:13px;color:#333;line-height:1.7;margin:0">{info["dataset_context"]}</p>
        </div>
        <div style="padding:14px 18px;background:#fafafa;border-top:1px solid #e0e0e0;">
            <div style="font-size:11px;font-weight:700;color:#1d9bd1;letter-spacing:0.8px;
                        margin-bottom:8px;">KEY POINTS</div>
            {bullets_html}
        </div>
        <div style="padding:12px 18px 14px;background:#f4f7ff;border-top:1px solid #e0e0e0;">
            <div style="font-size:11px;font-weight:700;color:#1d9bd1;letter-spacing:0.8px;
                        margin-bottom:6px;">REFERENCES AND FURTHER READING</div>
            {refs_html}
        </div>
    </div>
    """
    with teach_out:
        clear_output(wait=True)
        display(HTML(html))

# Teaching panel follows the MODE first (CrowdStrike / full get their own write-up),
# and the selected QUESTION when in the sample-stream mode.
_MODE_INFO_KEY = {"crowdstrike": "__crowdstrike__", "full": "__overview__"}

def refresh_teaching(change=None):
    k = MODES[mode_dd.value]["key"]
    if k in _MODE_INFO_KEY:
        _render_teaching_info(MODE_TEACHING[_MODE_INFO_KEY[k]])
        question_dd.disabled = (k == "full")   # full mode: question only picks which chart
    else:
        question_dd.disabled = False
        _render_teaching_info(TEACHING_CONTENT.get(question_dd.value))

mode_dd.observe(refresh_teaching, names="value")
question_dd.observe(refresh_teaching, names="value")
refresh_teaching()


In [6]:
# ==========================================================
# MAP HELPERS  (US-wide, per-airport)
# map_pdf is a per-airport aggregation with columns:
#   origin, city, lat, lon, flight_count, avg_dep_delay,
#   avg_arr_delay, pct_delayed
# ==========================================================
US_CENTER = [39.5, -98.35]
US_ZOOM   = 3

def add_legend(fmap, title, items):
    rows = ""
    for colour, label in items:
        if colour.startswith("grad:"):
            g = colour[5:]
            rows += (f"<div style='margin:4px 0'><div style='height:10px;width:100%;"
                     f"background:linear-gradient({g});border-radius:2px'></div>"
                     f"<span style='font-size:10px;color:#555'>{label}</span></div>")
        elif colour in ("dot-sm", "dot-lg"):
            sz = "8px" if colour == "dot-sm" else "14px"
            rows += (f"<div style='display:flex;align-items:center;margin:3px 0'>"
                     f"<div style='width:{sz};height:{sz};border-radius:50%;background:#888;"
                     f"margin-right:7px;flex-shrink:0'></div>"
                     f"<span style='font-size:11px;color:#555'>{label}</span></div>")
        else:
            rows += (f"<div style='display:flex;align-items:center;margin:3px 0'>"
                     f"<div style='width:13px;height:13px;background:{colour};margin-right:7px;"
                     f"flex-shrink:0;border-radius:2px'></div>"
                     f"<span style='font-size:11px;color:#555'>{label}</span></div>")
    legend_html = f"""
    <div style="position:fixed;top:16px;right:16px;background:rgba(255,255,255,0.95);
        border:1px solid #ccc;border-radius:6px;padding:8px 12px;font-family:Arial,sans-serif;
        z-index:9999;min-width:160px;box-shadow:1px 1px 5px rgba(0,0,0,0.15);">
    <div style='font-size:12px;font-weight:700;color:#0b2545;margin-bottom:5px'>{title}</div>
    {rows}</div>
    """
    fmap.get_root().html.add_child(folium.Element(legend_html))

def make_heatmap(map_pdf):
    m = folium.Map(location=US_CENTER, zoom_start=US_ZOOM, tiles="CartoDB dark_matter",
                   width=MAP_W_PX, height=int(MAP_H.replace("px", "")) - 12)
    pts = [[r["lat"], r["lon"], r["flight_count"]] for _, r in map_pdf.iterrows()]
    if pts:
        HeatMap(pts, radius=18, blur=22, min_opacity=0.35).add_to(m)
    add_legend(m, "Airport volume heatmap", [
        ("grad:to right, #00008B, #00FFFF, #FFFF00, #FF0000", "Low to high flight volume"),
    ])
    return m

def make_dot_map(map_pdf, value_col, label, is_minutes=False):
    cmap = cm.get_cmap("YlOrRd")
    m = folium.Map(location=US_CENTER, zoom_start=US_ZOOM, tiles="CartoDB positron",
                   width=MAP_W_PX, height=int(MAP_H.replace("px", "")) - 12)
    if len(map_pdf) == 0:
        return m
    vmin = map_pdf[value_col].quantile(0.05)
    vmax = map_pdf[value_col].quantile(0.95)
    if vmax <= vmin:
        vmax = vmin + 1
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    for _, row in map_pdf.iterrows():
        val    = row[value_col]
        colour = mcolors.to_hex(cmap(norm(val)))
        radius = 4 + 11 * float(np.clip(norm(val), 0, 1))
        suffix = " min" if is_minutes else ""
        folium.CircleMarker(
            location=[row["lat"], row["lon"]],
            radius=radius, color=colour, fill=True, fill_color=colour, fill_opacity=0.8,
            weight=0.5,
            tooltip=f"{row['city']} ({row['origin']})  {label}: {val:.1f}{suffix}"
        ).add_to(m)
    lo = f"{vmin:.0f}"; hi = f"{vmax:.0f}"
    add_legend(m, f"Dot map: {label}", [
        ("grad:to right, #FFFFB2, #FD8D3C, #BD0026", f"{lo} (yellow) to {hi} (red)"),
        ("dot-sm", f"Small dot = low {label}"),
        ("dot-lg", f"Large dot = high {label}"),
    ])
    return m

def make_combined_map(map_pdf):
    cmap = cm.get_cmap("YlOrRd")
    m = folium.Map(location=US_CENTER, zoom_start=US_ZOOM, tiles="CartoDB dark_matter",
                   width=MAP_W_PX, height=int(MAP_H.replace("px", "")) - 12)
    if len(map_pdf) == 0:
        return m
    pts = [[r["lat"], r["lon"], r["flight_count"]] for _, r in map_pdf.iterrows()]
    HeatMap(pts, radius=18, blur=22, min_opacity=0.25).add_to(m)
    dmin = map_pdf["avg_dep_delay"].quantile(0.05)
    dmax = map_pdf["avg_dep_delay"].quantile(0.95)
    if dmax <= dmin:
        dmax = dmin + 1
    dnorm = mcolors.Normalize(vmin=dmin, vmax=dmax)
    vmax_vol = map_pdf["flight_count"].quantile(0.95) or 1
    for _, row in map_pdf.iterrows():
        colour = mcolors.to_hex(cmap(dnorm(row["avg_dep_delay"])))
        radius = 4 + 10 * float(np.clip(row["flight_count"] / vmax_vol, 0, 1))
        folium.CircleMarker(
            location=[row["lat"], row["lon"]],
            radius=radius, color=colour, fill=True, fill_color=colour, fill_opacity=0.7,
            weight=0.5,
            tooltip=(f"{row['city']} ({row['origin']})  "
                     f"vol: {int(row['flight_count'])}  dep delay: {row['avg_dep_delay']:.1f} min")
        ).add_to(m)
    add_legend(m, "Combined map", [
        ("grad:to right, #00008B, #FF0000", "Heat: flight volume"),
        ("grad:to right, #FFFFB2, #BD0026", "Dot colour: avg dep delay"),
        ("dot-sm", "Small dot = low volume"),
        ("dot-lg", "Large dot = high volume"),
    ])
    return m


In [7]:
# ==========================================================
# CHART RENDERERS
# render_charts(q_label, pdf_window, batch_pdf, map_pdf)
#   pdf_window : per-window aggregation (accumulated)
#   batch_pdf  : raw rows of the CURRENT batch
#   map_pdf    : per-airport aggregation with coordinates
# ==========================================================
CAUSE_COLS   = ["carrier_delay", "weather_delay", "nas_delay",
                "security_delay", "late_aircraft_delay"]
CAUSE_LABELS = ["Carrier", "Weather", "NAS", "Security", "Late aircraft"]

def _style(ax):
    ax.set_facecolor("#f9f9f9")
    for sp in ax.spines.values():
        sp.set_edgecolor("#e0e0e0")

def _time_xticks(ax, pdf_window, n_ticks=6):
    """Label the x-axis with real window timestamps (e.g. 'Jul 19 06:00')
    instead of a bare window index, so a multi-day replay reads as a timeline."""
    if "win_label" not in pdf_window.columns or len(pdf_window) == 0:
        ax.set_xlabel("Window index"); return
    n = len(pdf_window)
    step = max(1, n // n_ticks)
    pos = list(range(0, n, step))
    ax.set_xticks(pos)
    ax.set_xticklabels([pdf_window["win_label"].iloc[i] for i in pos],
                       rotation=30, ha="right", fontsize=7)
    ax.set_xlabel("")

# carrier colours: Delta highlighted in alert red, peers muted
def _carrier_colours(labels):
    return ["#e03131" if str(c) == "DL" else "#4c6ef5" for c in labels]

def _flush_fig():
    """Render the current matplotlib figure to a PNG and display it explicitly.
    Unlike plt.show(), this appears reliably even when render runs on a
    background thread (the streaming loop), and closes the figure so figures
    do not accumulate. Must be called INSIDE a `with <output>:` block."""
    import io
    from IPython.display import Image
    fig = plt.gcf()
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=100, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    display(Image(data=buf.getvalue()))

def render_charts(q_label, pdf_window, batch_pdf, map_pdf):
    cat = QUESTION_MAP.get(q_label, "volume")
    ql  = q_label.lower()
    n   = len(pdf_window)
    x   = range(n)

    # ------------------------------ CHART A ------------------------------
    with chart1_out:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(5.5, 2.7)); fig.patch.set_facecolor("#fff"); _style(ax)

        if cat == "volume":
            if "rush" in ql and "dep_time_block" in batch_pdf.columns:
                order = ["Morning", "Afternoon", "Evening", "Overnight"]
                counts = batch_pdf["dep_time_block"].value_counts()
                counts = counts.reindex([o for o in order if o in counts.index])
                ax.bar(counts.index.astype(str), counts.values, color="#1d9bd1", width=0.6)
                ax.set_title("Departures by time-of-day block (batch)", fontsize=11, pad=8)
                ax.set_ylabel("Flights")
            else:
                ax.bar(x, pdf_window["flight_count"], color="#1d9bd1", width=0.7)
                ax.set_title("Flights per window over time", fontsize=11, pad=8)
                ax.set_ylabel("Flights"); _time_xticks(ax, pdf_window)

        elif cat == "delay":
            ax.plot(x, pdf_window["avg_dep_delay"], marker="o", color="#e8590c",
                    linewidth=1.8, markersize=3)
            ax.fill_between(x, pdf_window["avg_dep_delay"], alpha=0.12, color="#e8590c")
            ax.axhline(0, color="#adb5bd", linewidth=0.8)
            ax.set_title("Average departure delay over time", fontsize=11, pad=8)
            ax.set_ylabel("Dep delay (min)"); _time_xticks(ax, pdf_window)

        elif cat == "carrier":
            c = batch_pdf["op_unique_carrier"].value_counts().head(12)
            ax.bar(c.index.astype(str), c.values, color=_carrier_colours(c.index), width=0.65)
            ax.set_title("Flights per carrier (batch)", fontsize=11, pad=8)
            ax.set_ylabel("Flights"); ax.tick_params(axis="x", labelrotation=0)

        elif cat == "spatial":
            if len(map_pdf):
                if "worst" in ql:
                    t = map_pdf.sort_values("avg_dep_delay", ascending=False).head(10)
                    vals, title, xl = t["avg_dep_delay"], "Worst airports by avg dep delay", "Dep delay (min)"
                else:
                    t = map_pdf.sort_values("flight_count", ascending=False).head(10)
                    vals, title, xl = t["flight_count"], "Busiest airports by volume", "Flights"
                ax.barh(t["city"], vals, color="#1d9bd1")
                ax.invert_yaxis(); ax.set_title(title, fontsize=11, pad=8); ax.set_xlabel(xl)

        elif cat == "anomaly":
            if n > 1:
                thr = pdf_window["flight_count"].mean() + 1.5 * pdf_window["flight_count"].std()
            else:
                thr = pdf_window["flight_count"].iloc[0] if n else 0
            colours = ["#e03131" if v > thr else "#868e96" for v in pdf_window["flight_count"]]
            ax.bar(x, pdf_window["flight_count"], color=colours, width=0.7)
            ax.axhline(thr, color="#e03131", linestyle="--", linewidth=1.2,
                       label=f"Threshold = {thr:.0f}")
            ax.legend(fontsize=8)
            ax.set_title("Demand anomaly detection", fontsize=11, pad=8)
            ax.set_ylabel("Flights"); _time_xticks(ax, pdf_window)

        ax.tick_params(labelsize=8); plt.tight_layout(); _flush_fig()

    # ------------------------------ CHART B ------------------------------
    with chart2_out:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(5.5, 2.7)); fig.patch.set_facecolor("#fff"); _style(ax)

        if cat == "volume":
            ax.plot(x, pdf_window["active_airports"], marker="s", color="#2f9e44",
                    linewidth=1.8, markersize=3)
            ax.set_title("Active airports over time", fontsize=11, pad=8)
            ax.set_ylabel("Distinct origins"); _time_xticks(ax, pdf_window)

        elif cat == "delay":
            if "causes" in ql:
                totals = [float(batch_pdf[c].fillna(0).sum()) for c in CAUSE_COLS]
                order = np.argsort(totals)[::-1]
                labs = [CAUSE_LABELS[i] for i in order]; vals = [totals[i] for i in order]
                ax.bar(labs, vals, color="#1864ab", width=0.65)
                ax.set_title("Delay minutes by cause (batch)", fontsize=11, pad=8)
                ax.set_ylabel("Total minutes"); ax.tick_params(axis="x", labelrotation=20)
            elif "distance" in ql:
                s = batch_pdf.dropna(subset=["dep_delay", "distance"])
                s = s.sample(min(400, len(s))) if len(s) else s
                if len(s):
                    sc = ax.scatter(s["distance"], s["dep_delay"], c=s["air_time"],
                                    cmap="viridis", alpha=0.55, s=14)
                    plt.colorbar(sc, ax=ax, label="Air time (min)", shrink=0.8)
                ax.set_title("Departure delay vs route distance", fontsize=11, pad=8)
                ax.set_xlabel("Distance (mi)"); ax.set_ylabel("Dep delay (min)")
            else:
                ax2 = ax.twinx()
                ax.bar(x, pdf_window["avg_dep_delay"], color="#e8590c", alpha=0.5,
                       width=0.6, label="Avg dep delay")
                ax2.plot(x, pdf_window["avg_arr_delay"], color="#1864ab", marker="o",
                         linewidth=1.8, markersize=3, label="Avg arr delay")
                ax.set_ylabel("Dep delay (min)", color="#e8590c")
                ax2.set_ylabel("Arr delay (min)", color="#1864ab")
                ax.set_title("Departure vs arrival delay over time", fontsize=11, pad=8)
                _time_xticks(ax, pdf_window)
                h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
                ax.legend(h1 + h2, l1 + l2, fontsize=8)

        elif cat == "carrier":
            if "on-time" in ql:
                _op = batch_pdf[(batch_pdf["cancelled"] == 0) & (batch_pdf["diverted"] == 0)]
                g = (_op.groupby("op_unique_carrier")["is_delayed"]
                     .mean().mul(-100).add(100))           # on-time % of OPERATED flights
                g = g.sort_values(ascending=True).head(12)   # worst on-time first (Delta during outage)
                ax.bar(g.index.astype(str), g.values, color=_carrier_colours(g.index), width=0.65)
                ax.set_title("On-time rate by carrier (batch)", fontsize=11, pad=8)
                ax.set_ylabel("On-time %"); ax.set_ylim(0, 100)
            else:
                g = (batch_pdf.groupby("op_unique_carrier")["dep_delay"]
                     .mean().sort_values(ascending=False).head(12))
                ax.bar(g.index.astype(str), g.values, color=_carrier_colours(g.index), width=0.65)
                ax.set_title("Average dep delay by carrier (batch)", fontsize=11, pad=8)
                ax.set_ylabel("Dep delay (min)")

        elif cat == "spatial":
            if len(map_pdf):
                sizes = 20 + np.clip(map_pdf["pct_delayed"].fillna(0), 0, 100) * 1.2
                sc = ax.scatter(map_pdf["flight_count"], map_pdf["avg_dep_delay"],
                                s=sizes, c=map_pdf["avg_dep_delay"], cmap="YlOrRd",
                                alpha=0.7, edgecolors="#555", linewidths=0.3)
                plt.colorbar(sc, ax=ax, label="Dep delay (min)", shrink=0.8)
                ax.set_title("Airport volume vs delay (dot size = % delayed)", fontsize=10, pad=8)
                ax.set_xlabel("Flights"); ax.set_ylabel("Avg dep delay (min)")

        elif cat == "anomaly":
            if "cancellation" in ql:
                ax.bar(x, pdf_window["cancellations"], color="#c92a2a", width=0.7)
                ax.set_title("Cancellations over time", fontsize=11, pad=8)
                ax.set_ylabel("Cancellations"); _time_xticks(ax, pdf_window)
            else:
                d = batch_pdf["dep_delay"].dropna()
                if len(d):
                    thr = d.mean() + 2.5 * d.std()
                    ax.hist(d, bins=30, color="#868e96", edgecolor="white", linewidth=0.4)
                    ax.axvline(thr, color="#e03131", linestyle="--", linewidth=1.5,
                               label=f"Spike threshold = {thr:.0f} min")
                    ax.legend(fontsize=8)
                ax.set_title("Departure delay distribution", fontsize=11, pad=8)
                ax.set_xlabel("Dep delay (min)"); ax.set_ylabel("Count")

        ax.tick_params(labelsize=8); plt.tight_layout(); _flush_fig()

    # ------------------------------ MAPS ------------------------------
    map1_lbl.value = ("<p style='margin:6px 8px 2px;font-size:12px;font-weight:600;color:#444'>"
                      "Airport volume heatmap (accumulated)</p>")
    with map1_out:
        clear_output(wait=True)
        display(make_heatmap(map_pdf))

    if cat == "spatial" and ("heatmap +" in ql or "combined" in ql):
        map2_lbl.value = ("<p style='margin:6px 8px 2px;font-size:12px;font-weight:600;color:#444'>"
                          "Combined: volume heat + delay dots</p>")
        with map2_out:
            clear_output(wait=True); display(make_combined_map(map_pdf))
    elif cat == "delay" or (cat == "spatial" and "worst" in ql) or cat == "anomaly":
        map2_lbl.value = ("<p style='margin:6px 8px 2px;font-size:12px;font-weight:600;color:#444'>"
                          "Avg departure delay by airport</p>")
        with map2_out:
            clear_output(wait=True)
            display(make_dot_map(map_pdf, "avg_dep_delay", "Dep delay", is_minutes=True))
    else:
        map2_lbl.value = ("<p style='margin:6px 8px 2px;font-size:12px;font-weight:600;color:#444'>"
                          "Flight volume by airport</p>")
        with map2_out:
            clear_output(wait=True)
            display(make_dot_map(map_pdf, "flight_count", "Flights"))


In [8]:
# ==========================================================
# MAIN RUN LOOP — three modes, chosen from the Mode dropdown
#   crowdstrike : 50 batches x 500, real chronological order
#   sample      : 10 batches x 1000, time-of-day axis
#   full        : one-shot exact analysis of every row
#
# Convention (agreed after the data-quality audit):
#   % delayed  = delayed / OPERATED flights   (cancelled &
#                diverted excluded from the denominator)
#   cancellations reported separately, never as "not delayed"
# ==========================================================
_BATCH_COLS = ["op_unique_carrier", "dep_delay", "arr_delay", "is_delayed",
               "cancelled", "diverted", "distance", "air_time",
               "dep_time_block"] + CAUSE_COLS

def _airport_map_pdf(sdf):
    """Aggregate flights by origin and attach coordinates."""
    agg = (
        sdf.groupBy("origin")
        .agg(
            F.count("*").alias("flight_count"),
            F.round(F.avg("dep_delay"), 1).alias("avg_dep_delay"),
            F.round(F.avg("arr_delay"), 1).alias("avg_arr_delay"),
            F.round(100 * F.sum("is_delayed")
                    / F.greatest(F.sum("is_operated"), F.lit(1)), 1).alias("pct_delayed"),
        )
        .toPandas()
    )
    agg["lat"]  = agg["origin"].map(lambda a: AIRPORT_COORDS.get(a, (None, None, None))[0])
    agg["lon"]  = agg["origin"].map(lambda a: AIRPORT_COORDS.get(a, (None, None, None))[1])
    agg["city"] = agg["origin"].map(lambda a: AIRPORT_COORDS.get(a, (None, None, a))[2])
    agg["avg_dep_delay"] = agg["avg_dep_delay"].fillna(0)
    agg["avg_arr_delay"] = agg["avg_arr_delay"].fillna(0)
    return agg.dropna(subset=["lat", "lon"]).reset_index(drop=True)

def _window_pdf(sdf, window_dur, fmt):
    """Tumbling-window aggregation -> pandas frame with the agreed rate convention."""
    pdf = (
        sdf.groupBy(F.window("event_time", window_dur))
        .agg(
            F.count("*")                    .alias("flight_count"),
            F.sum("is_operated")            .alias("operated"),
            F.sum("is_delayed")             .alias("delayed"),
            F.round(F.avg("dep_delay"), 2)  .alias("avg_dep_delay"),
            F.round(F.avg("arr_delay"), 2)  .alias("avg_arr_delay"),
            F.round(F.avg("distance"), 0)   .alias("avg_distance"),
            F.countDistinct("origin")       .alias("active_airports"),
            F.sum("cancelled")              .alias("cancellations"),
        )
        .orderBy("window")
        .toPandas()
    )
    pdf["avg_dep_delay"] = pdf["avg_dep_delay"].fillna(0)
    pdf["avg_arr_delay"] = pdf["avg_arr_delay"].fillna(0)
    pdf["pct_delayed"]   = (100 * pdf["delayed"]
                            / pdf["operated"].replace(0, pd.NA)).fillna(0).round(1)
    pdf["pct_cancelled"] = (100 * pdf["cancellations"] / pdf["flight_count"]).round(1)
    pdf["win_start"]     = pdf["window"].apply(lambda w: w["start"])
    pdf["win_label"]     = pdf["win_start"].apply(lambda s: s.strftime(fmt))
    return pdf

def _render_kpis(tag, pdf, show_asof):
    total_flights = int(pdf["flight_count"].sum())
    total_oper    = int(pdf["operated"].sum())
    total_delayed = int(pdf["delayed"].sum())
    total_cancel  = int(pdf["cancellations"].sum())
    pct_delayed   = round(100 * total_delayed / total_oper, 1) if total_oper else 0.0
    pct_cancel    = round(100 * total_cancel / total_flights, 1) if total_flights else 0.0
    # arrival-delay average weighted by operated flights per window (not a mean of means)
    _wts = pdf["operated"].clip(lower=0)
    avg_arr = round(float((pdf["avg_arr_delay"] * _wts).sum() / _wts.sum()), 1) if _wts.sum() else 0.0
    date_card = ""
    if show_asof and pd.notna(pdf["win_start"].max()):
        date_card = f"""
            <div style="background:#0b2545;color:#fff;padding:10px 16px;border-radius:8px;min-width:120px;text-align:center">
                <div style="font-size:10px;opacity:.7;margin-bottom:2px">AS OF</div>
                <div style="font-size:22px;font-weight:700">{pdf["win_start"].max().strftime('%b %d %H:%M')}</div></div>"""
    with kpi_out:
        clear_output(wait=True)
        display(HTML(f"""
        <div style="display:flex;gap:10px;flex-wrap:wrap;font-family:Arial,sans-serif;padding:4px 0">
            <div style="background:#495057;color:#fff;padding:10px 16px;border-radius:8px;min-width:70px;text-align:center">
                <div style="font-size:10px;opacity:.7;margin-bottom:2px">{tag[0]}</div>
                <div style="font-size:22px;font-weight:700">{tag[1]}</div></div>
            {date_card}
            <div style="background:#1864ab;color:#fff;padding:10px 16px;border-radius:8px;min-width:90px;text-align:center">
                <div style="font-size:10px;opacity:.7;margin-bottom:2px">FLIGHTS</div>
                <div style="font-size:22px;font-weight:700">{total_flights:,}</div></div>
            <div style="background:#1d9bd1;color:#fff;padding:10px 16px;border-radius:8px;min-width:100px;text-align:center">
                <div style="font-size:10px;opacity:.7;margin-bottom:2px">AVG ARR DELAY</div>
                <div style="font-size:22px;font-weight:700">{avg_arr} min</div></div>
            <div style="background:#862e9c;color:#fff;padding:10px 16px;border-radius:8px;min-width:110px;text-align:center">
                <div style="font-size:10px;opacity:.7;margin-bottom:2px">% DELAYED (oper.)</div>
                <div style="font-size:22px;font-weight:700">{pct_delayed}%</div></div>
            <div style="background:#c92a2a;color:#fff;padding:10px 16px;border-radius:8px;min-width:90px;text-align:center">
                <div style="font-size:10px;opacity:.7;margin-bottom:2px">CANCELLED</div>
                <div style="font-size:22px;font-weight:700">{total_cancel:,}</div></div>
            <div style="background:#e03131;color:#fff;padding:10px 16px;border-radius:8px;min-width:95px;text-align:center">
                <div style="font-size:10px;opacity:.7;margin-bottom:2px">% CANCELLED</div>
                <div style="font-size:22px;font-weight:700">{pct_cancel}%</div></div>
        </div>
        """))

def _render_table(pdf):
    with table_out:
        clear_output(wait=True)
        show = pdf[["win_label", "flight_count", "avg_dep_delay", "avg_arr_delay",
                    "pct_delayed", "cancellations", "pct_cancelled"]].tail(12)
        display(show.rename(columns={
            "win_label": "window", "flight_count": "flights",
            "avg_dep_delay": "dep_delay", "avg_arr_delay": "arr_delay",
            "pct_delayed": "%_delayed(op)",
            "cancellations": "cancelled", "pct_cancelled": "%_cancelled"}))

# ==========================================================
# MAIN-THREAD run loop (renders reliably inline).
# The stream runs on the kernel thread, so the in-widget Stop
# button cannot interrupt mid-run — use the toolbar's
# "Interrupt the kernel" (the square stop icon) to halt a run.
# ==========================================================
def start_stream(b):
    global running
    running = True
    cfg  = MODES[mode_dd.value]
    mode = cfg["key"]

    # ---------------- FULL DATASET: one-shot exact analysis ----------------
    if mode == "full":
        status_lbl.value = "Analysing all rows..."
        try:
            pdf       = _window_pdf(full_source, cfg["window"], cfg["fmt"])
            batch_pdf = (full_source.select(*_BATCH_COLS)
                         .sample(withReplacement=False, fraction=0.001, seed=42)
                         .limit(5000).toPandas())
            map_pdf   = _airport_map_pdf(full_source)
            _render_kpis(("SCOPE", "ALL"), pdf, show_asof=False)
            render_charts(question_dd.value, pdf, batch_pdf, map_pdf)
            _render_table(pdf)
            status_lbl.value = "Complete (full dataset)"
        except Exception:
            import traceback
            status_lbl.value = "Error (see panel)"
            with table_out:
                clear_output(wait=True); print(traceback.format_exc())
        return

    # ---------------- STREAMS: replay batches ----------------
    source = cs_source if mode == "crowdstrike" else sample_source
    BATCH_SIZE, N_BATCHES, PAUSE = cfg["batch_size"], cfg["n_batches"], cfg["pause"]
    all_data = None
    try:
        for batch_id in range(N_BATCHES):
            if not running:
                break
            status_lbl.value = f"Batch {batch_id + 1} of {N_BATCHES}"

            new_batch = source.filter(
                (F.col("row_id") >= batch_id * BATCH_SIZE) &
                (F.col("row_id") <  (batch_id + 1) * BATCH_SIZE)
            )
            all_data = new_batch if all_data is None else all_data.union(new_batch)

            pdf       = _window_pdf(all_data, cfg["window"], cfg["fmt"])
            batch_pdf = new_batch.select(*_BATCH_COLS).toPandas()
            map_pdf   = _airport_map_pdf(all_data)

            _render_kpis(("BATCH", batch_id + 1), pdf, show_asof=(mode == "crowdstrike"))
            render_charts(question_dd.value, pdf, batch_pdf, map_pdf)
            _render_table(pdf)
            time.sleep(PAUSE)

        status_lbl.value = "Complete" if running else "Stopped"
    except KeyboardInterrupt:
        status_lbl.value = "Stopped (interrupted)"
    except Exception:
        import traceback
        status_lbl.value = "Error (see panel)"
        with table_out:
            clear_output(wait=True); print(traceback.format_exc())

run_btn.on_click(start_stream)
print("Dashboard ready. Pick a Mode and a Question above, then click the green button.")
print("To stop a running stream, use the toolbar's Interrupt (square) button.")


Dashboard ready. Pick a Mode and a Question above, then click the green button.
To stop a running stream, use the toolbar's Interrupt (square) button.
